[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-matthew-lam/relightable-3dgs/blob/main/training.ipynb)

# **Relightable 3DGS Notebook**

## 1. Checking the GPU / CUDA version

In [5]:
!nvidia-smi

## 2. Mounting Google Drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cloning repo and installing dependencies

In [25]:
import os

os.chdir("/content")  # always start from a fixed base, so this cell is safe to rerun in a live session
if not os.path.exists("/content/relightable-3dgs"):
    !git clone https://github.com/ml-matthew-lam/relightable-3dgs.git
%cd /content/relightable-3dgs
!git pull

In [8]:
!pip install -r requirements.txt

Restart runtime now, in case torch was somehow imported before installing `requirements.txt`.

## 4. Copying dataset from Drive to Colab's local disk

In [9]:
import os
import shutil

DRIVE_DATASET_ZIP = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkered_suzanne.zip"
LOCAL_DATASET_PATH = "/content/checkered_suzanne"

if not os.path.exists(LOCAL_DATASET_PATH):
    local_zip = "/content/checkered_suzanne.zip"
    shutil.copy(DRIVE_DATASET_ZIP, local_zip)
    shutil.unpack_archive(local_zip, "/content")
    os.remove(local_zip)
    print(f"copied and unzipped dataset to {LOCAL_DATASET_PATH}")
else:
    print(f"{LOCAL_DATASET_PATH} already exists, skipped copy")

## 5. Running the training script

In [ ]:
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkpoints/lambertian_week3"

!python train.py \
  --data_dir /content/checkered_suzanne \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --iters 15000 \
  --num_init_points 100000

If runtime disconnects mid-run: reconnect, re-run cells 1-4, then run the cell below instead of the cell above.

In [ ]:
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkpoints/lambertian_week3"

!python train.py \
  --data_dir /content/checkered_suzanne \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --iters 15000 \
  --num_init_points 100000 \
  --resume

## 6. Comparing renders

In [ ]:
# BEAUTY
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type beauty \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_beauty"

In [ ]:
# NORMALS
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type normals \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_normals"

In [ ]:
# ALBEDO
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type albedo \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_albedo"

In [ ]:
# BEAUTY - test light (light5)
!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_015000.pt" \
  --render_type beauty \
  --test_light \
  --out_dir "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_test_beauty"

## 7. Rendering and viewing animation/video

In [ ]:
import os

OUT_PATH = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/videos/video_demo.webp"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

!python render_video.py \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --data_dir "{LOCAL_DATASET_PATH}" \
  --num_camera_positions 6 \
  --camera_radius 4.5 \
  --camera_elevation_deg 20 \
  --light_radius 4.5 \
  --light_cycles 2 \
  --transition_frames 60 \
  --dwell_frames 90 \
  --fps 24 \
  --out "{OUT_PATH}"

In [ ]:
import base64
from IPython.display import HTML

with open(OUT_PATH, "rb") as f:
    webp_b64 = base64.b64encode(f.read()).decode("utf-8")
HTML(f'<img src="data:image/webp;base64,{webp_b64}">')